# Data Wrangling with Spark

This is the code used in the previous screencast. Run each code cell to understand what the code does and how it works.

These first three cells import libraries, instantiate a SparkSession, and then read in the data set

In [1]:
%use spark

In [2]:
fun Dataset<Row>.toHTML(limit: Int = 20, truncate: Int = 50): String {
    val sb = StringBuilder()
    sb.append("<html><body>")
    sb.append("<table><tr>")
    sb.append(schema().fieldNames().map { "<th style=\"text-align:left\">${it}</th>"}.joinToString(""))
    sb.append("</tr>")
    limit(limit).collectAsList().forEach { row ->
        sb.append("<tr>")
        (0 until row.size()).map {
            if (row[it] != null) row[it].toString() else ""
        }.forEach {
            val truncated = if (truncate > 0 && it.length > truncate) {
                if (truncate < 4) it.substring(0, truncate) else it.substring(0, truncate - 3) + "..."
            } else { it }
            sb.append("<td style=\"text-align:left\" title=\"$it\">$truncated</td>")
        }
        sb.append("</tr>")
    }
    sb.append("</table>")
    if(limit < count())
        sb.append("<p>... only showing top $limit rows</p>")
    sb.append("</body></html>")
    return sb.toString()
}

In [3]:
val path = "data/sparkify_log_small.json"
var userLog = spark.read().json(path)

# Data Exploration 

The next cells explore the data set.

In [4]:
userLog.takeAsList(5).forEach({it -> println(it)})

[Showaddywaddy,Logged In,Kenneth,M,112,Matthews,232.93342,paid,Charlotte-Concord-Gastonia, NC-SC,PUT,NextSong,1509380319284,5132,Christmas Tears Will Fall,200,1513720872284,"Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36",1046]
[Lily Allen,Logged In,Elizabeth,F,7,Chase,195.23873,free,Shreveport-Bossier City, LA,PUT,NextSong,1512718541284,5027,Cheryl Tweedy,200,1513720878284,"Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.143 Safari/537.36",1000]
[Cobra Starship Featuring Leighton Meester,Logged In,Vera,F,6,Blackwell,196.20526,paid,Racine, WI,PUT,NextSong,1499855749284,5516,Good Girls Go Bad (Feat.Leighton Meester) (Album Version),200,1513720881284,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.78.2 (KHTML, like Gecko) Version/7.0.6 Safari/537.78.2",2219]
[Alex Smoke,Logged In,Sophee,F,8,Barker,405.99465,paid,San Luis Obispo-Paso Robles-Arroyo Grande, CA,PUT,NextSong,1513009647

In [5]:
userLog.printSchema()

root
 |-- artist: string (nullable = true)
 |-- auth: string (nullable = true)
 |-- firstName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- itemInSession: long (nullable = true)
 |-- lastName: string (nullable = true)
 |-- length: double (nullable = true)
 |-- level: string (nullable = true)
 |-- location: string (nullable = true)
 |-- method: string (nullable = true)
 |-- page: string (nullable = true)
 |-- registration: long (nullable = true)
 |-- sessionId: long (nullable = true)
 |-- song: string (nullable = true)
 |-- status: long (nullable = true)
 |-- ts: long (nullable = true)
 |-- userAgent: string (nullable = true)
 |-- userId: string (nullable = true)



In [6]:
userLog.describe()

summary,artist,auth,firstName,gender,itemInSession,lastName,length,level,location,method,page,registration,sessionId,song,status,ts,userAgent,userId
count,8347,10000,9664,9664,10000,9664,8347,10000,9664,10000,10000,9664,10000,8347,10000,10000,9664,10000
mean,461.0,,,,19.6734,,249.6486587492503,,,,,1.5046953695887393E12,4436.7511,Infinity,202.8984,1.5137859954164E12,,1442.4413286423842
stddev,300.0,,,,25.382114916132597,,95.00437130781461,,,,,8.47314252131657E9,2043.1281541827557,NaN,18.041791154505876,3.2908288623601213E7,,829.8909432082613
min,!!!,Guest,Aakash,F,0,Acevedo,1.12281,free,"Aberdeen, WA",GET,About,1463503881284,9,#1,200,1513720872284,"""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10) ...",
max,ÃÂlafur Arnalds,Logged Out,Zoie,M,163,Zuniga,1806.8371,paid,"Yuma, AZ",PUT,Upgrade,1513760702284,7144,wingless,404,1513848349284,Mozilla/5.0 (compatible; MSIE 9.0; Windows NT 6...,999


In [7]:
userLog.describe("artist").show()

+-------+-----------------+
|summary|           artist|
+-------+-----------------+
|  count|             8347|
|   mean|            461.0|
| stddev|            300.0|
|    min|              !!!|
|    max|ÃÂlafur Arnalds|
+-------+-----------------+



In [8]:
userLog.describe("sessionId").show()

+-------+------------------+
|summary|         sessionId|
+-------+------------------+
|  count|             10000|
|   mean|         4436.7511|
| stddev|2043.1281541827557|
|    min|                 9|
|    max|              7144|
+-------+------------------+



In [9]:
userLog.count()

10000

In [10]:
userLog.select("page").dropDuplicates().sort("page").show()

+----------------+
|            page|
+----------------+
|           About|
|       Downgrade|
|           Error|
|            Help|
|            Home|
|           Login|
|          Logout|
|        NextSong|
|   Save Settings|
|        Settings|
|Submit Downgrade|
|  Submit Upgrade|
|         Upgrade|
+----------------+



In [11]:
userLog.select("userId", "firstname", "page", "song").where("userId == 1046").collectAsList().forEach({it -> println(it)})

[1046,Kenneth,NextSong,Christmas Tears Will Fall]
[1046,Kenneth,NextSong,Be Wary Of A Woman]
[1046,Kenneth,NextSong,Public Enemy No.1]
[1046,Kenneth,NextSong,Reign Of The Tyrants]
[1046,Kenneth,NextSong,Father And Son]
[1046,Kenneth,NextSong,No. 5]
[1046,Kenneth,NextSong,Seventeen]
[1046,Kenneth,Home,null]
[1046,Kenneth,NextSong,War on war]
[1046,Kenneth,NextSong,Killermont Street]
[1046,Kenneth,NextSong,Black & Blue]
[1046,Kenneth,Logout,null]
[1046,Kenneth,Home,null]
[1046,Kenneth,NextSong,Heads Will Roll]
[1046,Kenneth,NextSong,Bleed It Out [Live At Milton Keynes]]
[1046,Kenneth,NextSong,Clocks]
[1046,Kenneth,NextSong,Love Rain]
[1046,Kenneth,NextSong,Ry Ry's Song (Album Version)]
[1046,Kenneth,NextSong,The Invisible Man]
[1046,Kenneth,NextSong,Catch You Baby (Steve Pitron & Max Sanna Radio Edit)]
[1046,Kenneth,NextSong,Ask The Mountains]
[1046,Kenneth,NextSong,Given Up (Album Version)]
[1046,Kenneth,NextSong,El Cuatrero]
[1046,Kenneth,NextSong,Hero/Heroine]
[1046,Kenneth,NextSong,S

# Calculating Statistics by Hour

In [12]:
import org.apache.spark.sql.types.DataTypes
import java.time.Instant
import java.time.LocalDateTime
import java.time.ZoneId
import org.apache.spark.sql.api.java.UDF1


val hourFromTimestamp = UDF1<Long, Int> { ts -> LocalDateTime.ofInstant(Instant.ofEpochMilli(ts), ZoneId.of("Z")).hour }
spark.udf().register("hourFromTimestamp", hourFromTimestamp, DataTypes.IntegerType)

In [13]:
userLog = userLog.withColumn("hour", callUDF("hourFromTimestamp", userLog.col("ts")))

In [14]:
userLog.head()

[Showaddywaddy,Logged In,Kenneth,M,112,Matthews,232.93342,paid,Charlotte-Concord-Gastonia, NC-SC,PUT,NextSong,1509380319284,5132,Christmas Tears Will Fall,200,1513720872284,"Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36",1046,22]

In [15]:
var songsInHour = userLog.filter("page == 'NextSong'").groupBy("hour").count().orderBy("hour")

In [16]:
songsInHour.show()

+----+-----+
|hour|count|
+----+-----+
|   0|  456|
|   1|  454|
|   2|  382|
|   3|  302|
|   4|  352|
|   5|  276|
|   6|  348|
|   7|  358|
|   8|  375|
|   9|  249|
|  10|  216|
|  11|  228|
|  12|  251|
|  13|  339|
|  14|  462|
|  15|  479|
|  16|  484|
|  17|  430|
|  18|  362|
|  19|  295|
+----+-----+
only showing top 20 rows



In [17]:
%use krangl

In [18]:
val songsInHourKrangl = DataFrame.fromRecords(songsInHour.collectAsList(), {hc -> mapOf("hour" to hc[0], "count" to hc[1])})
songsInHourKrangl

hour,count
0,456
1,454
2,382
3,302
4,352
5,276
6,348
7,358
8,375
9,249


In [19]:
%use lets-plot

In [20]:
val df = mapOf(
    "hour" to songsInHourKrangl["hour"].asInts().toList()
    "count" to songsInHourKrangl["count"].asLongs().toList()
)
df

{hour=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23], count=[456, 454, 382, 302, 352, 276, 348, 358, 375, 249, 216, 228, 251, 339, 462, 479, 484, 430, 362, 295, 257, 248, 369, 375]}

In [21]:
var p = lets_plot(df) { x = "hour"; y = "count"}
p += geom_point()
p

# Drop Rows with Missing Values

As you'll see, it turns out there are no missing values in the userID or session columns. But there are userID values that are empty strings.

In [22]:
var userLogValid = userLog.na().drop("any", arrayOf("userId", "sessionId"))

In [23]:
userLogValid.count()

10000

In [24]:
userLog.select("userId").dropDuplicates().sort("userId").show()

+------+
|userId|
+------+
|      |
|    10|
|   100|
|  1000|
|  1003|
|  1005|
|  1006|
|  1017|
|  1019|
|  1020|
|  1022|
|  1025|
|  1030|
|  1035|
|  1037|
|   104|
|  1040|
|  1042|
|  1043|
|  1046|
+------+
only showing top 20 rows



In [25]:
userLogValid = userLogValid.filter("userId != ''")

In [26]:
userLogValid.count()

9664

# Users Downgrade Their Accounts

Find when users downgrade their accounts and then flag those log entries. Then use a window function and cumulative sum to distinguish each user's data as either pre or post downgrade events.

In [27]:
userLogValid.filter("page = 'Submit Downgrade'").show()

+------+---------+---------+------+-------------+--------+------+-----+--------------------+------+----------------+-------------+---------+----+------+-------------+--------------------+------+----+
|artist|     auth|firstName|gender|itemInSession|lastName|length|level|            location|method|            page| registration|sessionId|song|status|           ts|           userAgent|userId|hour|
+------+---------+---------+------+-------------+--------+------+-----+--------------------+------+----------------+-------------+---------+----+------+-------------+--------------------+------+----+
|  null|Logged In|    Kelly|     F|           24|  Newton|  null| paid|Houston-The Woodl...|   PUT|Submit Downgrade|1513283366284|     5931|null|   307|1513768454284|Mozilla/5.0 (Wind...|  1138|  11|
+------+---------+---------+------+-------------+--------+------+-----+--------------------+------+----------------+-------------+---------+----+------+-------------+--------------------+------+----+


In [28]:
userLog.select("userId", "firstname", "page", "level", "song").where("userId == 1138").collectAsList().forEach({it -> println(it)})

[1138,Kelly,Home,paid,null]
[1138,Kelly,NextSong,paid,Everybody Everybody]
[1138,Kelly,NextSong,paid,Gears]
[1138,Kelly,NextSong,paid,Use Somebody]
[1138,Kelly,NextSong,paid,Love Of My Life (1993 Digital Remaster)]
[1138,Kelly,NextSong,paid,Down In The Valley Woe]
[1138,Kelly,NextSong,paid,Treat Her Like A Lady]
[1138,Kelly,NextSong,paid,Everybody Thinks You're An Angel]
[1138,Kelly,NextSong,paid,Fourteen Wives]
[1138,Kelly,NextSong,paid,Love On The Rocks]
[1138,Kelly,NextSong,paid,Breakeven]
[1138,Kelly,NextSong,paid,Leaf House]
[1138,Kelly,NextSong,paid,NAISEN KANSSA]
[1138,Kelly,NextSong,paid,You're In My Heart]
[1138,Kelly,NextSong,paid,Roll On Down The Highway]
[1138,Kelly,NextSong,paid,Plasticities (Remix)]
[1138,Kelly,NextSong,paid,Secrets]
[1138,Kelly,NextSong,paid,Hello]
[1138,Kelly,NextSong,paid,I Never Told You]
[1138,Kelly,NextSong,paid,Love Break Me]
[1138,Kelly,NextSong,paid,One Touch One Bounce]
[1138,Kelly,NextSong,paid,Undo]
[1138,Kelly,NextSong,paid,Overdue (Blackbear

In [29]:
val flagDowngradeEvent = UDF1<String, Int> { page -> if (page == "Submit Downgrade") 1 else 0 }
spark.udf().register("flagDowngradeEvent", flagDowngradeEvent, DataTypes.IntegerType)

In [30]:
userLogValid = userLogValid.withColumn("downgraded", callUDF("flagDowngradeEvent", userLogValid.col("page")))

In [31]:
userLogValid.head()

[Showaddywaddy,Logged In,Kenneth,M,112,Matthews,232.93342,paid,Charlotte-Concord-Gastonia, NC-SC,PUT,NextSong,1509380319284,5132,Christmas Tears Will Fall,200,1513720872284,"Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36",1046,22,0]

In [32]:
import org.apache.spark.sql.expressions.Window

In [33]:
val windowval = Window.partitionBy("userId").orderBy(desc("ts")).rangeBetween(Window.unboundedPreceding(), 0)

In [34]:
userLogValid = userLogValid.withColumn("phase", sum("downgraded").over(windowval))

In [35]:
userLogValid.select("userId", "firstname", "ts", "page", "level", "phase").where("userId == 1138").sort("ts").collectAsList().forEach({it -> println(it})

[1138,Kelly,1513729066284,Home,paid,1]
[1138,Kelly,1513729066284,NextSong,paid,1]
[1138,Kelly,1513729313284,NextSong,paid,1]
[1138,Kelly,1513729552284,NextSong,paid,1]
[1138,Kelly,1513729783284,NextSong,paid,1]
[1138,Kelly,1513730001284,NextSong,paid,1]
[1138,Kelly,1513730263284,NextSong,paid,1]
[1138,Kelly,1513730518284,NextSong,paid,1]
[1138,Kelly,1513730768284,NextSong,paid,1]
[1138,Kelly,1513731182284,NextSong,paid,1]
[1138,Kelly,1513731435284,NextSong,paid,1]
[1138,Kelly,1513731695284,NextSong,paid,1]
[1138,Kelly,1513731857284,NextSong,paid,1]
[1138,Kelly,1513732160284,NextSong,paid,1]
[1138,Kelly,1513732302284,NextSong,paid,1]
[1138,Kelly,1513732540284,NextSong,paid,1]
[1138,Kelly,1513732770284,NextSong,paid,1]
[1138,Kelly,1513732994284,NextSong,paid,1]
[1138,Kelly,1513733223284,NextSong,paid,1]
[1138,Kelly,1513733456284,NextSong,paid,1]
[1138,Kelly,1513733738284,NextSong,paid,1]
[1138,Kelly,1513733941284,NextSong,paid,1]
[1138,Kelly,1513734289284,NextSong,paid,1]
[1138,Kelly,151